In [1]:
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests

base_url = "https://books.toscrape.com/catalogue/page-{}.html"
books_data = []

# غيّر عدد الصفحات حسب المطلوب في التاسك (مثلاً 1 إلى 50 لجمع 1000 كتاب، أو 1 إلى 10 وهكذا)
start_page = 1
end_page = 50

headers = {"User-Agent": "Mozilla/5.0"}

for page in range(start_page, end_page + 1):
    page_url = base_url.format(page)
    response = requests.get(page_url, headers=headers)

    if response.status_code != 200:
        print(f"Page {page} not accessible (status {response.status_code})")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.select("article.product_pod")

    for book in articles:
        # العنوان
        title_el = book.select_one("h3 a")
        title = (
            title_el.get("title", "").strip()
            if title_el and title_el.get("title")
            else "Unknown"
        )
        if not title:
            title = "Unknown"

        # السعر
        price_el = book.select_one("p.price_color")
        price = price_el.get_text(strip=True) if price_el else "Unknown"
        if not price:
            price = "Unknown"

        # حالة المخزون
        stock_el = book.select_one("p.instock.availability")
        in_stock = stock_el.get_text(" ", strip=True) if stock_el else "Unknown"
        if not in_stock:
            in_stock = "Unknown"

        # التصنيف من صفحة تفاصيل الكتاب
        category = "Unknown"
        if title_el and title_el.get("href"):
            book_url = urljoin(page_url, title_el["href"])
            try:
                book_res = requests.get(book_url, headers=headers, timeout=10)
                if book_res.status_code == 200:
                    book_soup = BeautifulSoup(book_res.text, "html.parser")
                    breadcrumb_items = book_soup.select("ul.breadcrumb li a")
                    if len(breadcrumb_items) >= 3:
                        cat_text = breadcrumb_items[2].get_text(strip=True)
                        if cat_text:
                            category = cat_text
            except Exception:
                category = "Unknown"

        books_data.append({
            "Category": category,
            "Title": title,
            "Price": price,
            "In_Stock": in_stock,
        })

    print(f"Scraped page {page}/{end_page} - Total books so far: {len(books_data)}")
    time.sleep(0.2)

df_scraped = pd.DataFrame(books_data)
df_scraped.to_csv("scraped_books.csv", index=False, encoding="utf-8-sig")
print("Done! Saved scraped_books.csv with shape:", df_scraped.shape)

Scraped page 1/50 - Total books so far: 20
Scraped page 2/50 - Total books so far: 40
Scraped page 3/50 - Total books so far: 60
Scraped page 4/50 - Total books so far: 80
Scraped page 5/50 - Total books so far: 100
Scraped page 6/50 - Total books so far: 120
Scraped page 7/50 - Total books so far: 140
Scraped page 8/50 - Total books so far: 160
Scraped page 9/50 - Total books so far: 180
Scraped page 10/50 - Total books so far: 200
Scraped page 11/50 - Total books so far: 220
Scraped page 12/50 - Total books so far: 240
Scraped page 13/50 - Total books so far: 260
Scraped page 14/50 - Total books so far: 280
Scraped page 15/50 - Total books so far: 300
Scraped page 16/50 - Total books so far: 320
Scraped page 17/50 - Total books so far: 340
Scraped page 18/50 - Total books so far: 360
Scraped page 19/50 - Total books so far: 380
Scraped page 20/50 - Total books so far: 400
Scraped page 21/50 - Total books so far: 420
Scraped page 22/50 - Total books so far: 440
Scraped page 23/50 - To